# Finding the fixtures in a flat's photos

A rental portal receives thousands of listing photos a day and wants to know what is in each flat
without anyone typing it in. This notebook finetunes `yolo11n` to find 27 fixtures in a room photo
(stove, fridge, toilet, bathtub, washing machine, air conditioning and so on), measures it class by
class, then prices one decision built on it.

The portal would use the detections in three ways:

- as search filters;
- to fill in listing fields the landlord left empty;
- to flag listings that claim an amenity no photo shows.

Each of those is a yes/no decision per fixture, with its own cost of being wrong. Most fixtures cost
nothing to miss, because every kitchen has a faucet. So one decision is worked through in full:
**does this photo show air conditioning?** AC is a real search filter, only about 31% of Spanish
homes have it, and landlords forget to tick the box.

Three numbers shape the design:

- the portal scores about **15,000 photos a day** (Madrid rentals);
- about **2% of listing photos show an AC unit**;
- a **moderator** can check any photo the model is unsure about, at EUR 0.04 a photo, but is not
  perfect either.

Run with `uv run python main.py train`. `SKIP_TRAIN=1` reuses `weights/best.pt` and reruns everything
after training, which is what to do after changing a cost. The dashboard reads
`artifacts/results.json` and `artifacts/test_scores.csv`.

## 0. Setup

The values under the dataset line are business assumptions, not modelling choices. Each one is
defended in REPORT.md.

In [ ]:
import json
import os
import shutil
import time
from collections import Counter, defaultdict
from math import sqrt
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

from api_key import load_api_key

WORKSPACE, PROJECT, VERSION = "so-d4hcz", "all_finalize", 3

POSITIVE_CLASSES = ["ac"]
COST_FN = 25.0  # EUR, a flat with AC hidden from AC searches
COST_FP = 5.0  # EUR, a flat without AC shown as having it
COST_TP = COST_TN = 0.0
COST_REVIEW = 0.04  # EUR per photo a moderator checks
BASE_RATE = 0.02  # share of real listing photos that show AC
REVIEWER_RECALL, REVIEWER_SPECIFICITY = 0.90, 0.97
PHOTOS_PER_DAY, SECONDS_PER_REVIEW, REVIEWER_HOURS_PER_DAY = 15_000, 10, 6.0

CFG = {
    "POSITIVE_CLASSES": POSITIVE_CLASSES, "COST_FN": COST_FN, "COST_FP": COST_FP,
    "COST_TP": COST_TP, "COST_TN": COST_TN, "COST_REVIEW": COST_REVIEW, "BASE_RATE": BASE_RATE,
    "REVIEWER_RECALL": REVIEWER_RECALL, "REVIEWER_SPECIFICITY": REVIEWER_SPECIFICITY,
    "PHOTOS_PER_DAY": PHOTOS_PER_DAY, "SECONDS_PER_REVIEW": SECONDS_PER_REVIEW,
    "REVIEWER_HOURS_PER_DAY": REVIEWER_HOURS_PER_DAY,
}

BASE_MODEL = "yolo11n.pt"
EPOCHS, IMGSZ, BATCH, SEED = 20, 640, 16, 0  # Ultralytics' defaults, except 20 epochs, not 100
DEFAULT_CONF = 0.25  # Ultralytics' predict default
SCORE_CONF = 0.001  # keep borderline boxes; the threshold is chosen where its cost is visible
THIN = 50  # fewer test instances than this is too few to trust
CUTS = tuple(round(0.05 * i, 2) for i in range(1, 20)) + (
    0.97, 0.98, 0.99, 0.993, 0.995, 0.997, 0.999, 0.9995, 0.9999,
)

# Our classes that COCO also has. Zero-shot, COCO scores 0 on all the others.
COCO_EQUIVALENT = {
    "bed": "bed", "dining_table": "dining table", "fridge": "refrigerator",
    "microwave": "microwave", "oven": "oven", "sink": "sink", "sofa": "couch",
    "toilet": "toilet", "tv": "tv",
}

SKIP_TRAIN = os.environ.get("SKIP_TRAIN") == "1"
DEVICE = "0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ROOT = Path.cwd()
DATA_DIR = ROOT / "datasets" / f"{PROJECT}-{VERSION}"
DATA_YAML = DATA_DIR / "data_local.yaml"
WEIGHTS = ROOT / "weights" / "best.pt"
ARTIFACTS = ROOT / "artifacts"
SAMPLES = ROOT / "samples" / "idealista"
RESULTS = {"config": CFG}


def normalise(name):
    return str(name).strip().lower().replace(" ", "_")


def images(split):
    return sorted((DATA_DIR / split / "images").glob("*.jpg"))


def read_labels(image):
    # Each line is a polygon, "class x1 y1 x2 y2 ...", in 0-1 units. Its share of the image is
    # the area of its bounding rectangle, which is what a detector is scored on.
    path = image.parent.parent / "labels" / f"{image.stem}.txt"
    found = []
    for line in path.read_text().splitlines() if path.exists() else []:
        if line.strip():
            c, *xy = line.split()
            xs, ys = [float(v) for v in xy[0::2]], [float(v) for v in xy[1::2]]
            found.append((int(c), (max(xs) - min(xs)) * (max(ys) - min(ys))))
    return found


def pct(x):
    return "-" if x is None or x != x else f"{x:.1%}"

## 1. The dataset

Roboflow Universe `so-d4hcz/all_finalize`, version 3. The version number is part of the id: it
freezes the images and the labels, so every number below can be reproduced. It is downloaded once and
cached in `datasets/`.

Look at the composition before training anything. Three things decide what the results can and cannot
say:

- **Test instances per class.** Below 50, a per-class number is too noisy to trust.
- **Object size.** This is the median box as a share of the image.
- **Leakage.** Roboflow names augmented copies `<source>_jpg.rf.<hash>.jpg`. A source photo that sits
  in both train and test turns the test into a memory test.

In [ ]:
if not DATA_DIR.exists():
    from roboflow import Roboflow

    rf = Roboflow(api_key=load_api_key())
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov11", location=str(DATA_DIR))

names = yaml.safe_load((DATA_DIR / "data.yaml").read_text())["names"]
CLASSES = [normalise(n) for n in (names.values() if isinstance(names, dict) else names)]

# Roboflow's data.yaml uses relative paths that break when Ultralytics runs from the repo root.
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR), "train": "train/images", "val": "valid/images", "test": "test/images",
    "names": dict(enumerate(CLASSES)),
}, sort_keys=False))
print(f"{len(CLASSES)} classes: {', '.join(CLASSES)}")

In [ ]:
def source_photo(image):
    return image.name.split(".rf.")[0].rsplit("_", 1)[0]


def audit():
    splits, seen_in = {}, defaultdict(set)
    for split in ("train", "valid", "test"):
        files, instances, photos, shares, empty = images(split), Counter(), Counter(), defaultdict(list), 0
        for image in files:
            boxes = read_labels(image)
            empty += not boxes
            for c, share in boxes:
                instances[CLASSES[c]] += 1
                shares[CLASSES[c]].append(share)
            for c in {c for c, _ in boxes}:
                photos[CLASSES[c]] += 1
            seen_in[source_photo(image)].add(split)
        splits[split] = {
            "images": len(files),
            "images_without_labels": empty,
            "classes": {
                c: {"instances": instances[c], "images": photos[c],
                    "median_share": float(np.median(shares[c])) if shares[c] else None}
                for c in CLASSES
            },
        }
    return {
        "dataset": f"{WORKSPACE}/{PROJECT}/{VERSION}",
        "url": f"https://universe.roboflow.com/{WORKSPACE}/{PROJECT}/dataset/{VERSION}",
        "classes": CLASSES,
        "splits": splits,
        "thin_test_classes": [c for c in CLASSES if splits["test"]["classes"][c]["instances"] < THIN],
        "leaked_sources": sum(1 for found in seen_in.values() if len(found) > 1),
    }


RESULTS["dataset"] = audit()
splits = RESULTS["dataset"]["splits"]
for split, s in splits.items():
    print(f"{split:<6}{s['images']:>6,} images, {s['images_without_labels']:,} with no labels")

print(f"\n{'class':<19}{'train':>7}{'valid':>7}{'test':>6}{'test photos':>13}{'median box':>12}")
for c in sorted(CLASSES, key=lambda c: -splits["test"]["classes"][c]["instances"]):
    counts = "".join(f"{splits[s]['classes'][c]['instances']:>7,}" for s in ("train", "valid"))
    test = splits["test"]["classes"][c]
    thin = "   thin" if c in RESULTS["dataset"]["thin_test_classes"] else ""
    print(f"{c:<19}{counts}{test['instances']:>6,}{test['images']:>13,}"
          f"{pct(splits['train']['classes'][c]['median_share']):>12}{thin}")
print(f"\nsource photos found in more than one split: {RESULTS['dataset']['leaked_sources']}")

## 2. Finetune

`yolo11n`, starting from its COCO checkpoint, trained on all 27 classes. It is a detector rather than
a classifier because most fixtures are a small part of the photo. AC is about 1% of it, so a
whole-image classifier would mostly be looking at walls.

The device is picked automatically: CUDA, then Apple's GPU, then CPU. The settings are saved with the
results, because the report has to say where and how the model was trained.

In [ ]:
if SKIP_TRAIN:
    print(f"SKIP_TRAIN=1: reusing {WEIGHTS.relative_to(ROOT)}")
    RESULTS["training"] = json.loads((ARTIFACTS / "results.json").read_text())["training"]
else:
    started = time.perf_counter()
    model = YOLO(BASE_MODEL)
    model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                device=DEVICE, project=str(ROOT / "runs"), name="finetune", exist_ok=True)
    WEIGHTS.parent.mkdir(exist_ok=True)
    shutil.copy(model.trainer.best, WEIGHTS)
    RESULTS["training"] = {
        "base_model": BASE_MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
        "device": torch.cuda.get_device_name(0) if DEVICE == "0" else DEVICE,
        "minutes": round((time.perf_counter() - started) / 60),
        "ultralytics": ultralytics.__version__, "torch": torch.__version__,
    }
print(RESULTS["training"])

## 3. Per-class evaluation on the test split

The test split gets two passes:

- **AP needs every box,** so the first pass keeps Ultralytics' validation default of 0.001.
- **A confusion matrix needs a working threshold,** so the second pass counts at 0.25. Its last row
  and column are background: a box where nothing was labelled, or a labelled object the model missed.

An average hides the class you care about, so the table is per class.

In [ ]:
model = YOLO(str(WEIGHTS))
run = {"data": str(DATA_YAML), "split": "test", "imgsz": IMGSZ, "device": DEVICE,
       "project": str(ROOT / "runs"), "exist_ok": True, "verbose": False}
metrics = model.val(**run, name="test", plots=False)
matrix = model.val(**run, name="test_matrix", conf=DEFAULT_CONF, plots=True).confusion_matrix.matrix.astype(int)

found = {CLASSES[c]: metrics.box.class_result(i) for i, c in enumerate(metrics.box.ap_class_index)}
per_class = []
for c in CLASSES:
    p, r, ap50, ap = (float(v) for v in found[c]) if c in found else (None,) * 4
    n = splits["test"]["classes"][c]["instances"]
    per_class.append({"class": c, "test_instances": n, "precision": p, "recall": r,
                      "ap50": ap50, "ap50_95": ap, "thin": n < THIN})

print(f"{'class':<19}{'test':>6}{'precision':>11}{'recall':>8}{'AP50':>7}{'AP50-95':>9}")
for row in sorted(per_class, key=lambda r: -1 if r["ap50"] is None else r["ap50"], reverse=True):
    if row["ap50"] is None:
        print(f"{row['class']:<19}{row['test_instances']:>6}   not in the test split")
        continue
    print(f"{row['class']:<19}{row['test_instances']:>6}{row['precision']:>11.1%}{row['recall']:>8.1%}"
          f"{row['ap50']:>7.3f}{row['ap50_95']:>9.3f}" + ("   thin" if row["thin"] else ""))
overall = {"map50": float(metrics.box.map50), "map50_95": float(metrics.box.map)}
print(f"\nmAP50 {overall['map50']:.3f}, mAP50-95 {overall['map50_95']:.3f}")

labels = CLASSES + ["background"]
mistakes = sorted(
    ((int(matrix[p, t]), labels[t], labels[p])
     for p in range(len(labels)) for t in range(len(labels)) if p != t and matrix[p, t]),
    reverse=True,
)
print(f"\nlargest confusions at confidence {DEFAULT_CONF}:")
for count, true, predicted in mistakes[:8]:
    print(f"{count:>6}  true {true:<19} predicted {predicted}")

RESULTS["per_class"] = {"overall": overall, "classes": per_class}
RESULTS["confusion_matrix"] = {"labels": labels, "conf": DEFAULT_CONF, "matrix": matrix.tolist()}

## 4. Turning boxes into one score per photo

The model returns boxes, but the portal needs one number per photo and fixture. The rule bridging them
is a design decision, not a modelling one:

> **score** = confidence of the most confident box of that fixture in the photo, or 0 if there is none.

Taking the maximum rather than the mean is deliberate. One clear AC unit is enough for the flat to
have AC, and averaging it against a vague second box would dilute exactly the evidence that matters.

Every test photo is predicted at confidence 0.001, so borderline cases survive to be priced. The COCO
model is scored the same way, through `COCO_EQUIVALENT`, for the baseline in section 5. The result is
one row per photo in `artifacts/test_scores.csv`, which is what the dashboard's sliders recompute
costs from.

In [ ]:
def scores(model, image, rename):
    result = model.predict(str(image), conf=SCORE_CONF, imgsz=IMGSZ, device=DEVICE,
                           classes=list(rename), verbose=False)[0]
    best = dict.fromkeys(CLASSES, 0.0)
    for c, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
        best[rename[int(c)]] = max(best[rename[int(c)]], conf)
    return best


coco = YOLO(BASE_MODEL)
coco_ids = {name: i for i, name in coco.names.items()}
OWN = {i: normalise(name) for i, name in model.names.items()}
VIA_COCO = {coco_ids[theirs]: ours for ours, theirs in COCO_EQUIVALENT.items()}

started = time.perf_counter()
records = []
for image in images("test"):
    labelled = {CLASSES[c] for c, _ in read_labels(image)}
    ours, theirs = scores(model, image, OWN), scores(coco, image, VIA_COCO)
    records.append({
        "path": image.relative_to(ROOT).as_posix(),
        **{f"truth_{c}": c in labelled for c in CLASSES},
        **{f"score_{c}": ours[c] for c in CLASSES},
        **{f"coco_{c}": theirs[c] for c in CLASSES},
    })
ARTIFACTS.mkdir(exist_ok=True)
pd.DataFrame(records).to_csv(ARTIFACTS / "test_scores.csv", index=False)
elapsed = time.perf_counter() - started
print(f"{len(records)} test photos scored by both models in {elapsed:.0f}s "
      f"({elapsed / len(records):.2f}s per photo)")

## 5. Baselines on the same test photos

From here on the decision is the costed one: **does this photo show air conditioning?** A photo is
positive if its label file holds an `ac` box. Three systems are compared, all at Ultralytics' default
cut of 0.25:

- the **finetuned** model;
- **COCO zero-shot**, the pretrained `yolo11n.pt`. COCO has no air-conditioning class, so it scores 0
  on every photo. The table shows that instead of dropping the row;
- **always "no"**, which never matches. It is also the majority class, because few test photos show
  AC.

Be suspicious of accuracy. "Always no" scores well on it by finding nothing. Recall carries a 95%
Wilson interval, because a recall measured on a handful of positive photos is wide.

Then comes the same image-level comparison for every fixture. Where COCO has the class, it is a real
competitor.

In [ ]:
def decision_rows(records, classes, prefix="score"):
    # Several classes can stand for one request. A photo shows it if it shows any of them.
    return [
        {"path": r["path"],
         "truth": "positive" if any(r[f"truth_{c}"] for c in classes) else "negative",
         "score": max(r[f"{prefix}_{c}"] for c in classes)}
        for r in records
    ]


def ratio(a, b):
    return a / b if b else float("nan")


def confusion(rows, cut):
    verdicts = [("positive" if r["score"] >= cut else "negative", r["truth"]) for r in rows]
    tp = sum(1 for v, t in verdicts if v == "positive" and t == "positive")
    fn = sum(1 for v, t in verdicts if v == "negative" and t == "positive")
    fp = sum(1 for v, t in verdicts if v == "positive" and t == "negative")
    tn = sum(1 for v, t in verdicts if v == "negative" and t == "negative")
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "recall": ratio(tp, tp + fn), "specificity": ratio(tn, tn + fp),
            "precision": ratio(tp, tp + fp), "accuracy": ratio(tp + tn, len(verdicts))}


def wilson(successes, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = successes / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, centre - half), min(1.0, centre + half)


ROWS = decision_rows(records, POSITIVE_CLASSES)
positives = sum(r["truth"] == "positive" for r in ROWS)
systems = {
    "finetuned yolo11n": ROWS,
    "COCO zero-shot": decision_rows(records, POSITIVE_CLASSES, "coco"),
    "always no": [{**r, "score": 0.0} for r in ROWS],
}
print(f"{'+'.join(POSITIVE_CLASSES)} in {positives} of {len(ROWS)} test photos ({positives / len(ROWS):.1%})\n")
print(f"{'system':<20}{'recall':>8}{'95% CI':>12}{'specificity':>13}{'precision':>11}{'accuracy':>10}")
baselines = []
for name, rows in systems.items():
    m = confusion(rows, DEFAULT_CONF)
    ci = wilson(m["tp"], m["tp"] + m["fn"])
    baselines.append({"system": name, "cut": DEFAULT_CONF, **m, "recall_ci": ci})
    print(f"{name:<20}{pct(m['recall']):>8}{f'[{ci[0]:.0%}, {ci[1]:.0%}]':>12}"
          f"{pct(m['specificity']):>13}{pct(m['precision']):>11}{pct(m['accuracy']):>10}")

print(f"\n{'fixture':<19}{'photos':>7}{'recall':>8}{'specificity':>13}{'COCO recall':>13}{'COCO specif.':>14}")
vs_coco = []
for c in CLASSES:
    ours = confusion(decision_rows(records, [c]), DEFAULT_CONF)
    theirs = confusion(decision_rows(records, [c], "coco"), DEFAULT_CONF) if c in COCO_EQUIVALENT else None
    vs_coco.append({"class": c, "photos": ours["tp"] + ours["fn"],
                    "recall": ours["recall"], "specificity": ours["specificity"],
                    "coco_recall": theirs and theirs["recall"], "coco_specificity": theirs and theirs["specificity"]})
    coco_text = f"{pct(theirs['recall']):>13}{pct(theirs['specificity']):>14}" if theirs else f"{'no class':>13}"
    print(f"{c:<19}{ours['tp'] + ours['fn']:>7}{pct(ours['recall']):>8}{pct(ours['specificity']):>13}{coco_text}")

RESULTS["baselines"] = {"positive_classes": POSITIVE_CLASSES, "test_photos": len(ROWS),
                        "test_positives": positives, "systems": baselines, "per_class": vs_coco}

## 6. The base rate changes precision

The test split has AC in about 6% of photos, but real listings have it in about 2%. Recall and
specificity are properties of the model and do not move when the mix changes. Precision does, because
it depends on how many photos without AC there are to be wrong about.

So every cost from here on is projected onto `BASE_RATE`, never read off the test counts.

In [ ]:
def project(recall, specificity, base_rate, n=1000):
    tp = base_rate * recall * n
    fn = base_rate * (1 - recall) * n
    fp = (1 - base_rate) * (1 - specificity) * n
    return {"fn": fn, "fp": fp, "precision": ratio(tp, tp + fp)}


m = confusion(ROWS, DEFAULT_CONF)
print(f"measured at cut {DEFAULT_CONF}: recall {m['recall']:.1%}, specificity {m['specificity']:.1%}\n")
print(f"{'base rate':>10}{'precision':>11}{'FN':>7}{'FP':>7}")
for rate in (positives / len(ROWS), 0.20, 0.05, BASE_RATE, 0.005):
    p = project(m["recall"], m["specificity"], rate)
    print(f"{rate:>10.1%}{p['precision']:>11.1%}{p['fn']:>7.1f}{p['fp']:>7.1f}")
print("\nper 1,000 photos. Recall and specificity are identical in every row.")

## 7. Cost against threshold

A miss (FN) costs EUR 25 and a false match (FP) EUR 5. So a cut is judged by what it costs per 1,000
photos at the real base rate, not by accuracy or F1, which weigh the two errors equally.

With no moderator, every photo is either matched or not. The cost function below already allows for a
review band between two thresholds. A single cut is just the band with `low == high`, so section 8
reuses it unchanged.

In [ ]:
def zone_rates(rows, low, high):
    # How often a photo of this kind lands in each zone: not matched, review, matched.
    n = len(rows)
    return (ratio(sum(r["score"] < low for r in rows), n),
            ratio(sum(low <= r["score"] < high for r in rows), n),
            ratio(sum(r["score"] >= high for r in rows), n))


def three_way_cost(rows, low, high, cfg=CFG):
    # Per 1,000 photos at the real base rate. Zone rates are measured per truth class, so they
    # belong to the model and survive the shift in base rate.
    pos_miss, pos_review, _ = zone_rates([r for r in rows if r["truth"] == "positive"], low, high)
    _, neg_review, neg_match = zone_rates([r for r in rows if r["truth"] == "negative"], low, high)
    pos_n, neg_n = 1000 * cfg["BASE_RATE"], 1000 * (1 - cfg["BASE_RATE"])
    fn = pos_n * (pos_miss + pos_review * (1 - cfg["REVIEWER_RECALL"]))
    fp = neg_n * (neg_match + neg_review * (1 - cfg["REVIEWER_SPECIFICITY"]))
    reviews = pos_n * pos_review + neg_n * neg_review
    cost = (fn * cfg["COST_FN"] + fp * cfg["COST_FP"] + (pos_n - fn) * cfg["COST_TP"]
            + (neg_n - fp) * cfg["COST_TN"] + reviews * cfg["COST_REVIEW"])
    return {"fn": fn, "fp": fp, "reviews": reviews, "reviewed_share": reviews / 1000, "cost": cost}


single = []
print(f"{'cut':>7}{'recall':>8}{'specificity':>13}{'accuracy':>10}{'FN':>7}{'FP':>7}{'EUR/1,000':>11}")
for cut in CUTS:
    m, d = confusion(ROWS, cut), three_way_cost(ROWS, cut, cut)
    single.append({"cut": cut, "recall": m["recall"], "specificity": m["specificity"], "cost": d["cost"]})
    print(f"{cut:>7}{m['recall']:>8.1%}{m['specificity']:>13.1%}{m['accuracy']:>10.1%}"
          f"{d['fn']:>7.1f}{d['fp']:>7.1f}{d['cost']:>11,.2f}")

best = min(single, key=lambda s: s["cost"])
m = confusion(ROWS, best["cut"])
ci = wilson(m["tp"], m["tp"] + m["fn"])
print(f"\ncheapest cut {best['cut']}: EUR {best['cost']:,.2f} per 1,000, "
      f"recall {m['recall']:.1%} [{ci[0]:.0%}, {ci[1]:.0%}]")

## 8. Routing uncertain photos to a moderator

A single cut forces a yes or a no on every photo. That throws away what the score is good at: saying
when the model is unsure. Two thresholds give three outcomes:

| score | outcome |
|---|---|
| `>= high` | **matched**: the listing shows up in AC searches |
| `low <= score < high` | **review**: a moderator checks the photo |
| `< low` | **not matched** |

The moderator is not an oracle. A review costs EUR 0.04 and swaps the model's error rates for the
moderator's (90% recall, 97% specificity). So review only pays where the moderator is better than the
model.

Every band on the grid is priced the same way as the single cut. That includes the two extremes, "a
moderator checks everything" and "the model decides alone". The cheapest band is the operating point,
and the daily volume turns its review share into a number of people.

In [ ]:
def rank_designs(rows, cfg=CFG):
    # 0.0 and 1.01 let a band reach photos with no box at all, so "check everything" competes too.
    edges = (0.0, *CUTS, 1.01)
    options = [(three_way_cost(rows, low, high, cfg), low, high)
               for i, low in enumerate(edges) for high in edges[i:]]
    return sorted(options, key=lambda option: option[0]["cost"])


def staffing(reviews_per_1000, cfg=CFG):
    per_day = reviews_per_1000 / 1000 * cfg["PHOTOS_PER_DAY"]
    hours = per_day * cfg["SECONDS_PER_REVIEW"] / 3600
    return {"reviews_per_day": per_day, "review_hours_per_day": hours,
            "moderators": hours / cfg["REVIEWER_HOURS_PER_DAY"]}


ranked = rank_designs(ROWS)
print(f"{'low':>7}{'high':>7}{'reviewed':>10}{'FN':>7}{'FP':>7}{'EUR/1,000':>11}")
for d, low, high in ranked[:5]:
    print(f"{low:>7}{high:>7}{d['reviewed_share']:>10.1%}{d['fn']:>7.2f}{d['fp']:>7.2f}{d['cost']:>11,.2f}")

_, LOW, HIGH = ranked[0]
designs = {}
for name, (low, high) in {
    "model + moderator band": (LOW, HIGH),
    "model alone": (best["cut"], best["cut"]),
    "moderator checks everything": (0.0, 1.01),
    "never match": (1.01, 1.01),
    "match everything": (0.0, 0.0),
}.items():
    d = three_way_cost(ROWS, low, high)
    designs[name] = {"low": low, "high": high, **d, **staffing(d["reviews"])}

print(f"\n{'design':<29}{'band':>15}{'reviewed':>10}{'EUR/1,000':>11}{'reviews/day':>13}{'moderators':>12}")
for name, d in designs.items():
    band = f"[{d['low']}, {d['high']})"
    print(f"{name:<29}{band:>15}{d['reviewed_share']:>10.1%}{d['cost']:>11,.2f}"
          f"{d['reviews_per_day']:>13,.0f}{d['moderators']:>12.2f}")

Every number above rests on guesses: the costs, the base rate and the moderator's recall. So each one
is moved while the rest stay put, to find the point where the recommendation flips.

The last list reverses the question. It shows the photos without an AC label that the model is surest
about. Open them. "The model is wrong" and "the label is wrong" look identical in a table, and only
one of them is fixed by a better model.

In [ ]:
sensitivity = []
print(f"{'assumption':<17}{'value':>8}{'best band':>16}{'reviewed':>10}{'EUR/1,000':>11}")
for name, values in {
    "COST_FN": [COST_FN * k for k in (0.25, 0.5, 1, 2, 4)],
    "BASE_RATE": [BASE_RATE * k for k in (0.25, 0.5, 1, 2, 4)],
    "COST_REVIEW": [COST_REVIEW * k for k in (0.25, 0.5, 1, 2, 4)],
    "REVIEWER_RECALL": [0.5, 0.7, 0.9, 0.99],
}.items():
    for value in values:
        d, low, high = rank_designs(ROWS, {**CFG, name: value})[0]
        sensitivity.append({"parameter": name, "value": value, "low": low, "high": high,
                            "reviewed_share": d["reviewed_share"], "cost": d["cost"]})
        band = f"[{low}, {high})"
        print(f"{name:<17}{value:>8.4g}{band:>16}{d['reviewed_share']:>10.1%}{d['cost']:>11,.2f}")

print("\nphotos without an AC label, highest score first:")
for r in sorted((r for r in ROWS if r["truth"] == "negative"), key=lambda r: -r["score"])[:5]:
    print(f"  {r['score']:.3f}  {r['path']}")

op = designs["model + moderator band"]
print(f"\noperating point [{LOW}, {HIGH}): EUR {op['cost']:,.2f} per 1,000, {op['reviewed_share']:.1%} "
      f"reviewed, {op['reviews_per_day']:,.0f} reviews a day, {op['moderators']:.2f} moderators")
RESULTS["operating_point"] = op
RESULTS["sweep"] = {"single": single, "designs": designs, "sensitivity": sensitivity}

## 9. Idealista check

The test split comes from the same Hong Kong listings as the training data. A photo from a real
Spanish listing shows whether the model still finds the fixtures there. Put photos in
`samples/idealista/`; they are never trained on. Every fixture the model finds is listed with its
outcome under the chosen band.

In [ ]:
def outcome(score, low=LOW, high=HIGH):
    return "matched" if score >= high else "review" if score >= low else "not matched"


photos = sorted(p for p in SAMPLES.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}) \
    if SAMPLES.exists() else []
idealista = []
for photo in photos:
    found = {c: s for c, s in scores(model, photo, OWN).items() if s >= LOW}
    idealista.append({"path": photo.relative_to(ROOT).as_posix(), "fixtures": found})
    print(photo.name)
    for c, s in sorted(found.items(), key=lambda item: -item[1]):
        print(f"  {c:<19}{s:>7.3f}  {outcome(s)}")
if not photos:
    print("No photos in samples/idealista/, so there is nothing to check.")
RESULTS["idealista"] = idealista

## Save

Everything the dashboard reads goes into one file, next to `test_scores.csv`.

In [ ]:
(ARTIFACTS / "results.json").write_text(json.dumps(RESULTS, indent=2))
print("wrote artifacts/results.json and artifacts/test_scores.csv")